# Estructuras de datos probabilísticas · MinHash, reservorio, skip list y hashing consistente

Más estructuras y algoritmos que usan el azar (o un hash que se comporta como tal) para ganar memoria o tiempo: **MinHash** y **LSH** para estimar la similitud de Jaccard y encontrar casi duplicados sin comparar todos los pares, **muestreo de reservorio** para quedarse con una muestra uniforme de un flujo de longitud desconocida (comprobado con la prueba chi-cuadrado), la **skip list**, una lista ordenada con atajos aleatorios que compite con un árbol equilibrado, y el **hashing consistente** para repartir claves entre servidores moviendo las mínimas al añadir o quitar uno. Como en el notebook anterior, las funciones hash independientes salen de `blake2b` con sal y todas las semillas son fijas.

## 1. MinHash: estimar la similitud de Jaccard

La similitud de Jaccard de dos conjuntos es $|A \cap B| / |A \cup B|$. Si a cada elemento se le aplica una función hash, la probabilidad de que el mínimo de los dos conjuntos coincida es exactamente esa similitud; con `k` funciones (sales) se obtiene una *firma* de `k` mínimos y la fracción de posiciones iguales estima la similitud con un error típico de $\sqrt{J(1-J)/k}$. Representar un texto por sus *tejas* (grupos de 3 palabras seguidas), crear 20 variantes cambiando cada vez más palabras y comparar la similitud exacta con la estimada con firmas de 16, 64 y 256 valores.

In [1]:
import bisect
import hashlib
import heapq
import math
import random
from collections import Counter
from itertools import combinations, islice

import numpy as np
from scipy import stats


def hash64(x, sal=0):
    datos = hashlib.blake2b(str(x).encode(), digest_size=8, salt=sal.to_bytes(16, "little")).digest()
    return int.from_bytes(datos, "little")


TEXTO = """el alisio sopla del nordeste y deja un mar de nubes sobre las medianías del norte mientras el sur
de la isla sigue despejado y caluroso durante casi todo el año los barrancos bajan desde la cumbre hasta la
costa y en sus laderas crecen tabaibas cardones y palmeras que aguantan meses sin lluvia en invierno la nieve
cubre a veces el pico más alto y desde la playa se ve blanco sobre el azul del cielo""".split()
VOCABULARIO = sorted(set(TEXTO)) + ["volcán", "lava", "guagua", "papas", "mojo", "gofio", "charco", "risco"]


def variante(palabras, fraccion, rng):
    copia = list(palabras)
    for i in rng.sample(range(len(copia)), round(fraccion * len(copia))):
        copia[i] = rng.choice(VOCABULARIO)
    return copia


def tejas(palabras, ancho=3):
    return {" ".join(palabras[i:i + ancho]) for i in range(len(palabras) - ancho + 1)}


def jaccard(a, b):
    return len(a & b) / len(a | b)


def firma(conjunto, k, sal=0):
    return [min(hash64(x, sal + i) for x in conjunto) for i in range(k)]


def similitud(f1, f2):
    return sum(a == b for a, b in zip(f1, f2)) / len(f1)


rng = random.Random(1)
original = tejas(TEXTO)
firma_original = firma(original, 256)
fracciones = np.linspace(0.02, 0.9, 20)
errores = {16: [], 64: [], 256: []}
print(f"{'cambiadas':>9} {'exacta':>7} {'k=16':>6} {'k=64':>6} {'k=256':>6}")
for i, fraccion in enumerate(fracciones):
    otra = tejas(variante(TEXTO, fraccion, rng))
    f = firma(otra, 256)
    exacta = jaccard(original, otra)
    estimadas = {k: similitud(firma_original[:k], f[:k]) for k in errores}
    for k in errores:
        errores[k].append(abs(estimadas[k] - exacta))
    if i % 4 == 0:
        print(f"{fraccion:9.0%} {exacta:7.3f} " + " ".join(f"{estimadas[k]:6.3f}" for k in errores))
    assert abs(estimadas[256] - exacta) <= 4 * math.sqrt(exacta * (1 - exacta) / 256) + 0.01
print("Error absoluto medio:", {k: round(float(np.mean(v)), 4) for k, v in errores.items()})
assert np.mean(errores[256]) < np.mean(errores[64]) < np.mean(errores[16])

cambiadas  exacta   k=16   k=64  k=256
       2%   0.852  0.875  0.891  0.832
      21%   0.327  0.375  0.422  0.379
      39%   0.103  0.125  0.156  0.141
      58%   0.014  0.000  0.031  0.023
      76%   0.014  0.000  0.016  0.020


Error absoluto medio: {16: 0.0427, 64: 0.0301, 256: 0.0142}


## 2. LSH: buscar casi duplicados sin comparar todos los pares

Con firmas de $k = b \cdot r$ valores, el *locality-sensitive hashing* parte cada firma en `b` bandas de `r` filas y mete cada banda en una tabla: dos documentos son candidatos si coinciden en alguna banda entera, lo que ocurre con probabilidad $1 - (1 - s^r)^b$, una curva en S alrededor de $(1/b)^{1/r}$. Generar 120 documentos al azar más 40 casi duplicados (con tejas de 2 palabras), buscar los candidatos con `b = 20` y `r = 5`, y compararlos con la fuerza bruta sobre todos los pares: cuántas comparaciones se ahorran y qué proporción de pares similares se encuentra según su similitud.

In [2]:
rng = random.Random(2)
palabras = [f"palabra{i}" for i in range(400)]
documentos = [[rng.choice(palabras) for _ in range(50)] for _ in range(120)]
for _ in range(40):
    base, cambio = rng.choice(documentos[:120]), rng.uniform(0.01, 0.25)
    documentos.append([rng.choice(palabras) if rng.random() < cambio else p for p in base])
conjuntos = [tejas(d, ancho=2) for d in documentos]

b, r = 20, 5
firmas = [firma(c, b * r, sal=500) for c in conjuntos]
cubetas = {}
for i, f in enumerate(firmas):
    for banda in range(b):
        cubetas.setdefault((banda, tuple(f[banda * r:(banda + 1) * r])), []).append(i)
candidatos = {par for grupo in cubetas.values() for par in combinations(grupo, 2)}

todos = list(combinations(range(len(documentos)), 2))
similitudes = {par: jaccard(conjuntos[par[0]], conjuntos[par[1]]) for par in todos}
print(f"Pares totales: {len(todos)}; candidatos LSH: {len(candidatos)} ({len(candidatos) / len(todos):.2%})")
print(f"Umbral aproximado (1/b)^(1/r) = {(1 / b) ** (1 / r):.2f}\n")
print(f"{'similitud':>11} {'pares':>6} {'encontrados':>12} {'teórico':>8}")
for bajo, alto in [(0, 0.3), (0.3, 0.45), (0.45, 0.55), (0.55, 0.65), (0.65, 0.8), (0.8, 1.01)]:
    pares = [par for par, s in similitudes.items() if bajo <= s < alto]
    if pares:
        hallados = sum(par in candidatos for par in pares) / len(pares)
        teorico = np.mean([1 - (1 - similitudes[par] ** r) ** b for par in pares])
        print(f"{bajo:4.2f} – {min(alto, 1):4.2f} {len(pares):6} {hallados:12.1%} {teorico:8.1%}")
parecidos = [par for par, s in similitudes.items() if s >= 0.7]
assert sum(par in candidatos for par in parecidos) / len(parecidos) >= 0.95
assert len(candidatos) < 0.05 * len(todos)

Pares totales: 12720; candidatos LSH: 24 (0.19%)
Umbral aproximado (1/b)^(1/r) = 0.55

  similitud  pares  encontrados  teórico
0.00 – 0.30  12678         0.0%     0.0%
0.30 – 0.45     10        10.0%    19.9%
0.45 – 0.55      8        12.5%    44.0%
0.55 – 0.65      9        88.9%    76.7%
0.65 – 0.80      7        85.7%    96.9%
0.80 – 1.00      8       100.0%   100.0%


## 3. Muestreo de reservorio

Escribir `reservorio(flujo, k, rng)` (algoritmo R): guarda los `k` primeros elementos y, al llegar el elemento número `i` (contando desde 0), lo mete en la muestra con probabilidad $k/(i+1)$ sustituyendo a uno al azar. Así, en cualquier momento, cada elemento visto tiene la misma probabilidad de estar en la muestra, sin conocer la longitud del flujo ni guardarlo. Probarlo con un generador de llegadas de vuelos de longitud desconocida y con un flujo más corto que `k`.

In [3]:
def reservorio(flujo, k, rng):
    muestra = []
    for i, x in enumerate(flujo):
        if i < k:
            muestra.append(x)
        else:
            j = rng.randrange(i + 1)
            if j < k:
                muestra[j] = x
    return muestra


def llegadas(rng):
    vuelo = 0
    while rng.random() > 0.0005:
        vuelo += 1
        yield f"vuelo {vuelo:05d} a {rng.choice(['TFN', 'TFS', 'LPA', 'ACE', 'FUE', 'SPC', 'VDE', 'GMZ'])}"


rng = random.Random(3)
muestra = reservorio(llegadas(random.Random(30)), 5, rng)
todas = list(llegadas(random.Random(30)))
print(f"El flujo tenía {len(todas)} llegadas; muestra de 5:")
for linea in muestra:
    print(" ", linea)
assert len(muestra) == 5 and set(muestra) <= set(todas) and len(set(muestra)) == 5
assert sorted(reservorio(range(3), 5, rng)) == [0, 1, 2]

El flujo tenía 1065 llegadas; muestra de 5:
  vuelo 00321 a GMZ
  vuelo 00991 a FUE
  vuelo 00801 a LPA
  vuelo 00882 a TFS
  vuelo 00116 a LPA


## 4. Uniformidad con chi-cuadrado

Repetir 20 000 veces el muestreo de 5 elementos de un flujo de 20 y contar cuántas veces sale cada uno. Si el algoritmo es uniforme, cada elemento debe salir unas $20\,000 \cdot 5/20 = 5000$ veces. Calcular el estadístico chi-cuadrado a mano, comprobarlo con `scipy.stats.chisquare` y aceptar la uniformidad si el p-valor no es pequeño. Hacer lo mismo con una versión con un error típico, `randrange(i)` en lugar de `randrange(i + 1)`, y ver que la prueba la rechaza.

In [4]:
def reservorio_erroneo(flujo, k, rng):
    muestra = []
    for i, x in enumerate(flujo):
        if i < k:
            muestra.append(x)
        else:
            j = rng.randrange(i)
            if j < k:
                muestra[j] = x
    return muestra


def chi_cuadrado(observados, esperados):
    return sum((o - e) ** 2 / e for o, e in zip(observados, esperados))


repeticiones, n, k = 20_000, 20, 5
for nombre, algoritmo in (("correcto", reservorio), ("con error", reservorio_erroneo)):
    rng = random.Random(4)
    cuentas = Counter(x for _ in range(repeticiones) for x in algoritmo(range(n), k, rng))
    observados = [cuentas[i] for i in range(n)]
    esperados = [repeticiones * k / n] * n
    prueba = stats.chisquare(observados, esperados)
    print(f"{nombre:9}: primeros {observados[:4]}… últimos {observados[-3:]}; chi² = {prueba.statistic:.1f}, p = {prueba.pvalue:.3g}")
    assert math.isclose(prueba.statistic, chi_cuadrado(observados, esperados))
    assert (prueba.pvalue > 0.01) == (nombre == "correcto")

correcto : primeros [4961, 5085, 5007, 5014]… últimos [5020, 4966, 5090]; chi² = 15.8, p = 0.673
con error: primeros [4128, 4323, 4232, 4236]… últimos [5291, 5280, 5221]; chi² = 841.0, p = 4.01e-166


## 5. Algoritmo L: saltar en lugar de sortear

El algoritmo R sortea un número por cada elemento del flujo. El algoritmo L calcula directamente cuántos elementos saltar hasta el siguiente que entra en la muestra (con una distribución geométrica), así que solo hace $O(k(1 + \log(n/k)))$ sorteos. Implementarlo sobre un iterador con `itertools.islice`, contar los sorteos de ambos con un flujo de un millón de elementos y comprobar también su uniformidad con chi-cuadrado.

In [5]:
def abierto(rng):
    u = rng.random()
    while u == 0:
        u = rng.random()
    return u


def reservorio_l(flujo, k, rng):
    it = iter(flujo)
    muestra = list(islice(it, k))
    sorteos = 0
    if len(muestra) < k:
        return muestra, sorteos
    w = math.exp(math.log(abierto(rng)) / k)
    sorteos += 1
    while True:
        salto = math.floor(math.log(abierto(rng)) / math.log(1 - w))
        sorteos += 1
        siguiente = next(islice(it, salto, salto + 1), None)
        if siguiente is None:
            return muestra, sorteos
        muestra[rng.randrange(k)] = siguiente
        w *= math.exp(math.log(abierto(rng)) / k)
        sorteos += 2


class Contado(random.Random):
    sorteos = 0

    def randrange(self, *args):
        self.sorteos += 1
        return super().randrange(*args)


rng = Contado(5)
muestra_r = reservorio(range(1_000_000), 10, rng)
muestra_l, sorteos_l = reservorio_l(range(1_000_000), 10, random.Random(5))
print(f"Algoritmo R: {rng.sorteos} sorteos → {sorted(muestra_r)}")
print(f"Algoritmo L: {sorteos_l} sorteos → {sorted(muestra_l)}")
print(f"Reemplazos esperados k(1 + ln(n/k)) ≈ {10 * (1 + math.log(1_000_000 / 10)):.0f}, con unos 3 sorteos cada uno")
assert sorteos_l < 1000 < rng.sorteos

rng = random.Random(6)
cuentas = Counter(x for _ in range(20_000) for x in reservorio_l(range(20), 5, rng)[0])
p = stats.chisquare([cuentas[i] for i in range(20)]).pvalue
print(f"Uniformidad del algoritmo L: p = {p:.3f}")
assert p > 0.01

Algoritmo R: 999990 sorteos → [160580, 251060, 326105, 342940, 397503, 523349, 564070, 699660, 705572, 959549]
Algoritmo L: 353 sorteos → [62267, 140378, 228152, 312411, 319984, 507657, 572386, 608916, 917287, 941873]
Reemplazos esperados k(1 + ln(n/k)) ≈ 125, con unos 3 sorteos cada uno
Uniformidad del algoritmo L: p = 0.308


## 6. Reservorio ponderado

Para que cada elemento entre con probabilidad proporcional a un peso, el algoritmo de Efraimidis y Spirakis asigna a cada uno la clave $u^{1/w}$ (con `u` uniforme) y se queda con las `k` claves mayores usando un montículo de tamaño `k`. Con `k = 1`, comprobar con chi-cuadrado que la frecuencia de cada elemento es proporcional a su peso; después, sacar una muestra de 3 puestos de un mercado agrícola con pesos inventados.

In [6]:
def reservorio_ponderado(flujo, k, rng):
    monticulo = []
    for x, peso in flujo:
        clave = math.log(abierto(rng)) / peso
        if len(monticulo) < k:
            heapq.heappush(monticulo, (clave, x))
        elif clave > monticulo[0][0]:
            heapq.heapreplace(monticulo, (clave, x))
    return [x for _, x in sorted(monticulo, reverse=True)]


pesos = {"a": 1, "b": 2, "c": 3, "d": 4, "e": 5, "f": 6}
rng = random.Random(7)
repeticiones = 30_000
cuentas = Counter(reservorio_ponderado(pesos.items(), 1, rng)[0] for _ in range(repeticiones))
total = sum(pesos.values())
esperados = [repeticiones * p / total for p in pesos.values()]
observados = [cuentas[x] for x in pesos]
p = stats.chisquare(observados, esperados).pvalue
print("Observados:", observados)
print("Esperados: ", [round(e) for e in esperados], f"p = {p:.3f}")
assert p > 0.01

puestos = {"papas": 40, "plátanos": 35, "quesos": 20, "mojos": 10, "flores": 5, "artesanía": 3, "miel": 8}
print("Muestra ponderada de 3 puestos:", reservorio_ponderado(puestos.items(), 3, random.Random(8)))

Observados: [1436, 2878, 4406, 5703, 7016, 8561]
Esperados:  [1429, 2857, 4286, 5714, 7143, 8571] p = 0.321
Muestra ponderada de 3 puestos: ['miel', 'plátanos', 'mojos']


## 7. Skip list: búsqueda e inserción

Una skip list es una lista enlazada ordenada en la que cada nodo tiene una altura aleatoria (sube un nivel con probabilidad `p = 1/2`) y un enlace por nivel; los niveles altos son atajos que se saltan muchos nodos. Buscar consiste en avanzar por el nivel más alto mientras la siguiente clave sea menor y bajar un nivel cuando no se puede. Implementar `SkipList` con búsqueda (`in`) e inserción (sin repetidos), contando las comparaciones, y dibujar los niveles de una lista pequeña.

In [7]:
class NodoSkip:
    __slots__ = ("clave", "siguientes")

    def __init__(self, clave, nivel):
        self.clave = clave
        self.siguientes = [None] * nivel


class SkipList:
    def __init__(self, p=0.5, max_nivel=32, semilla=0):
        self.p, self.max_nivel = p, max_nivel
        self.rng = random.Random(semilla)
        self.cabeza = NodoSkip(None, max_nivel)
        self.nivel = 1
        self.n = 0
        self.comparaciones = 0

    def nivel_aleatorio(self):
        nivel = 1
        while nivel < self.max_nivel and self.rng.random() < self.p:
            nivel += 1
        return nivel

    def predecesores(self, clave):
        anteriores = [self.cabeza] * self.max_nivel
        x = self.cabeza
        for i in reversed(range(self.nivel)):
            while (s := x.siguientes[i]) is not None:
                self.comparaciones += 1
                if not s.clave < clave:
                    break
                x = s
            anteriores[i] = x
        return anteriores

    def __contains__(self, clave):
        x = self.predecesores(clave)[0].siguientes[0]
        return x is not None and x.clave == clave

    def insertar(self, clave):
        anteriores = self.predecesores(clave)
        x = anteriores[0].siguientes[0]
        if x is not None and x.clave == clave:
            return False
        nodo = NodoSkip(clave, self.nivel_aleatorio())
        self.nivel = max(self.nivel, len(nodo.siguientes))
        for i in range(len(nodo.siguientes)):
            nodo.siguientes[i] = anteriores[i].siguientes[i]
            anteriores[i].siguientes[i] = nodo
        self.n += 1
        return True

    def nodos(self):
        x = self.cabeza.siguientes[0]
        while x is not None:
            yield x
            x = x.siguientes[0]

    def __iter__(self):
        return (x.clave for x in self.nodos())

    def __len__(self):
        return self.n

    def dibujar(self):
        nodos = list(self.nodos())
        ancho = max(len(str(x.clave)) for x in nodos) + 1
        for i in reversed(range(self.nivel)):
            fila = "".join(f"{x.clave:>{ancho}}" if len(x.siguientes) > i else "-" * ancho for x in nodos)
            print(f"nivel {i + 1}: {fila}")


lista = SkipList(semilla=3)
claves = random.Random(9).sample(range(100), 16)
for c in claves:
    lista.insertar(c)
lista.dibujar()
assert list(lista) == sorted(claves)
assert all(c in lista for c in claves) and not any(c in lista for c in range(100) if c not in claves)
assert not lista.insertar(claves[0]) and len(lista) == 16

nivel 3: --------- 17--- 34--------------------------- 89
nivel 2:   0  5--- 17 23 34 42------ 59--------- 78 86 89
nivel 1:   0  5 10 17 23 34 42 43 47 59 64 70 77 78 86 89


## 8. Borrado y comparación con `bisect`

Añadir `borrar` (que desengancha el nodo en todos sus niveles y baja la altura de la lista si quedan niveles vacíos) y `rango(a, b)`, que recorre las claves de `[a, b)` empezando por el predecesor de `a`. Aplicar 30 000 operaciones aleatorias (insertar, borrar, buscar y consultar rangos) a la vez a la skip list y a una lista ordenada mantenida con `bisect`, comprobando que dan siempre las mismas respuestas.

In [8]:
class SkipListCompleta(SkipList):
    def borrar(self, clave):
        anteriores = self.predecesores(clave)
        x = anteriores[0].siguientes[0]
        if x is None or x.clave != clave:
            return False
        for i in range(len(x.siguientes)):
            anteriores[i].siguientes[i] = x.siguientes[i]
        while self.nivel > 1 and self.cabeza.siguientes[self.nivel - 1] is None:
            self.nivel -= 1
        self.n -= 1
        return True

    def rango(self, a, b):
        x = self.predecesores(a)[0].siguientes[0]
        while x is not None and x.clave < b:
            yield x.clave
            x = x.siguientes[0]


rng = random.Random(10)
skip, ordenada = SkipListCompleta(semilla=10), []
operaciones = Counter()
for _ in range(30_000):
    op, clave = rng.choice(["insertar", "insertar", "borrar", "buscar", "rango"]), rng.randrange(5000)
    i = bisect.bisect_left(ordenada, clave)
    presente = i < len(ordenada) and ordenada[i] == clave
    match op:
        case "insertar":
            assert skip.insertar(clave) == (not presente)
            if not presente:
                ordenada.insert(i, clave)
        case "borrar":
            assert skip.borrar(clave) == presente
            if presente:
                ordenada.pop(i)
        case "buscar":
            assert (clave in skip) == presente
        case "rango":
            fin = clave + rng.randrange(200)
            assert list(skip.rango(clave, fin)) == ordenada[i:bisect.bisect_left(ordenada, fin)]
    operaciones[op] += 1
assert list(skip) == ordenada and len(skip) == len(ordenada)
print(dict(operaciones), f"→ {len(ordenada)} claves al final, altura {skip.nivel}")

{'rango': 5996, 'buscar': 5974, 'insertar': 11905, 'borrar': 6125} → 3229 claves al final, altura 13


## 9. Altura y número de nodos por nivel

En una skip list de `n` nodos con `p = 1/2`, el número de nodos que llegan al nivel `i` sigue una binomial de media $n\,p^{i-1}$, la altura ronda $\log_2 n$ y una búsqueda hace del orden de $\log_{1/p} n / p$ comparaciones. Construir una con $2^{15}$ claves y comparar los nodos por nivel con lo esperado, la altura con $\log_2 n$, y las comparaciones medias de 2000 búsquedas con las de la búsqueda binaria de `bisect` ($\log_2 n$).

In [9]:
n = 2 ** 15
skip = SkipListCompleta(semilla=11)
for c in random.Random(11).sample(range(10 * n), n):
    skip.insertar(c)
alturas = Counter(len(x.siguientes) for x in skip.nodos())
print(f"{'nivel':>5} {'nodos':>6} {'esperado':>9}")
for i in range(1, skip.nivel + 1):
    al_menos = sum(v for h, v in alturas.items() if h >= i)
    esperado = n * 0.5 ** (i - 1)
    print(f"{i:5} {al_menos:6} {esperado:9.1f}")
    assert abs(al_menos - esperado) <= 5 * math.sqrt(esperado) + 1
print(f"Altura: {skip.nivel}; log2(n) = {math.log2(n):.0f}")
assert math.log2(n) - 2 <= skip.nivel <= math.log2(n) + 8

claves = list(skip)
busquedas = random.Random(12).sample(claves, 2000)
skip.comparaciones = 0
assert all(c in skip for c in busquedas)
print(f"Comparaciones por búsqueda: skip list {skip.comparaciones / 2000:.1f}, "
      f"teórico log2(n)/p = {math.log2(n) / 0.5:.0f}, bisect ≈ {math.log2(n):.0f}")

nivel  nodos  esperado
    1  32768   32768.0
    2  16429   16384.0
    3   8207    8192.0
    4   4140    4096.0
    5   2105    2048.0
    6   1049    1024.0
    7    561     512.0
    8    297     256.0
    9    146     128.0
   10     73      64.0
   11     38      32.0
   12     24      16.0
   13     12       8.0
   14      8       4.0
   15      6       2.0
   16      2       1.0
   17      1       0.5
Altura: 17; log2(n) = 15
Comparaciones por búsqueda: skip list 29.5, teórico log2(n)/p = 30, bisect ≈ 15


## 10. Efecto de la probabilidad `p`

Con `p` más pequeña los nodos son más bajos: menos enlaces por nodo (en media $1/(1-p)$) pero más pasos horizontales en cada nivel. Construir skip lists de $2^{14}$ claves con `p` = 1/2, 1/4, 1/8 y 1/16 y comparar los enlaces medios por nodo, la altura y las comparaciones medias por búsqueda.

In [10]:
n = 2 ** 14
claves = random.Random(13).sample(range(10 * n), n)
busquedas = random.Random(14).sample(claves, 2000)
print(f"{'p':>6} {'enlaces/nodo':>13} {'1/(1-p)':>8} {'altura':>7} {'comparaciones':>14}")
for p in (1 / 2, 1 / 4, 1 / 8, 1 / 16):
    skip = SkipListCompleta(p=p, semilla=15)
    for c in claves:
        skip.insertar(c)
    enlaces = sum(len(x.siguientes) for x in skip.nodos()) / n
    skip.comparaciones = 0
    assert all(c in skip for c in busquedas)
    print(f"1/{round(1 / p):<4} {enlaces:13.3f} {1 / (1 - p):8.3f} {skip.nivel:7} {skip.comparaciones / 2000:14.1f}")
    assert abs(enlaces * (1 - p) - 1) < 0.03

     p  enlaces/nodo  1/(1-p)  altura  comparaciones
1/2            2.004    2.000      17           26.9


1/4            1.331    1.333       7           24.3
1/8            1.143    1.143       6           29.3
1/16           1.067    1.067       5           43.5


## 11. Anillo de hashing consistente

Para repartir claves entre servidores, el hashing consistente coloca cada servidor en varios puntos de un anillo de $2^{64}$ posiciones (sus *nodos virtuales*, hash de `"servidor#v"`) y asigna cada clave al primer punto que encuentra avanzando desde su hash. Implementar `AnilloConsistente` con una lista ordenada y `bisect`, y comprobar con 3000 claves que coincide con la búsqueda por fuerza bruta del punto más cercano en el sentido del anillo. Al quitar un servidor, solo deben cambiar de sitio sus claves; al añadir uno, solo deben moverse claves hacia él.

In [11]:
ESPACIO = 2 ** 64
SERVIDORES = ["tenerife", "gran-canaria", "lanzarote", "fuerteventura", "la-palma",
              "la-gomera", "el-hierro", "la-graciosa", "lobos", "alegranza", "montaña-clara"]


class AnilloConsistente:
    def __init__(self, virtuales=100, sal=7):
        self.virtuales, self.sal = virtuales, sal
        self.puntos = []

    def añadir(self, servidor, peso=1):
        for v in range(self.virtuales * peso):
            bisect.insort(self.puntos, (hash64(f"{servidor}#{v}", self.sal), servidor))

    def quitar(self, servidor):
        self.puntos = [p for p in self.puntos if p[1] != servidor]

    def servidor(self, h):
        i = bisect.bisect_left(self.puntos, (h, ""))
        return self.puntos[i % len(self.puntos)][1]

    def servidor_fuerza_bruta(self, h):
        return min(self.puntos, key=lambda p: (p[0] - h) % ESPACIO)[1]


claves = [f"sesion-{i}" for i in range(20_000)]
hashes = [hash64(c, 99) for c in claves]

anillo = AnilloConsistente()
for s in SERVIDORES[:10]:
    anillo.añadir(s)
antes = [anillo.servidor(h) for h in hashes]
assert all(anillo.servidor(h) == anillo.servidor_fuerza_bruta(h) for h in hashes[:3000])
print("Reparto con 10 servidores:", dict(Counter(antes).most_common(3)), "…")

anillo.quitar("lanzarote")
sin_lanzarote = [anillo.servidor(h) for h in hashes]
movidas = [(a, d) for a, d in zip(antes, sin_lanzarote) if a != d]
print(f"Al quitar lanzarote se mueven {len(movidas)} claves, todas suyas: {all(a == 'lanzarote' for a, _ in movidas)}")
assert all(a == "lanzarote" for a, _ in movidas) and len(movidas) == antes.count("lanzarote")

anillo.añadir("lanzarote")
assert [anillo.servidor(h) for h in hashes] == antes
anillo.añadir(SERVIDORES[10])
despues = [anillo.servidor(h) for h in hashes]
assert all(d == SERVIDORES[10] for a, d in zip(antes, despues) if a != d)

Reparto con 10 servidores: {'fuerteventura': 2321, 'tenerife': 2160, 'la-palma': 2134} …
Al quitar lanzarote se mueven 2093 claves, todas suyas: True


## 12. Módulo `N` frente a hashing consistente

La forma ingenua de repartir es `hash % N`, pero al pasar de `N` a `N + 1` servidores casi todas las claves cambian de servidor (en media una fracción $N/(N+1)$), lo que vacía todas las cachés a la vez. Con el anillo solo se mueve la parte que le toca al nuevo, alrededor de $1/(N+1)$. Medirlo al pasar de 10 a 11 servidores.

In [12]:
def reparto_modulo(n):
    return [SERVIDORES[h % n] for h in hashes]


movidas_modulo = sum(a != d for a, d in zip(reparto_modulo(10), reparto_modulo(11))) / len(hashes)
movidas_anillo = sum(a != d for a, d in zip(antes, despues)) / len(hashes)
print(f"{'método':22} {'movidas':>8} {'esperado':>9}")
print(f"{'hash % N':22} {movidas_modulo:8.1%} {10 / 11:9.1%}")
print(f"{'anillo (100 virtuales)':22} {movidas_anillo:8.1%} {1 / 11:9.1%}")
assert movidas_modulo > 0.85 and movidas_anillo < 0.15

método                  movidas  esperado
hash % N                  91.0%     90.9%
anillo (100 virtuales)     9.1%      9.1%


## 13. Nodos virtuales y pesos

Con un solo punto por servidor, los arcos del anillo son muy desiguales y algunos servidores reciben varias veces más claves que otros. Medir la carga máxima respecto a la media y el coeficiente de variación con 1, 10, 100 y 1000 nodos virtuales por servidor. Después, dar a cada servidor un número de nodos virtuales proporcional a su capacidad y comprobar que la fracción de claves que recibe se ajusta a su peso.

In [13]:
print(f"{'virtuales':>9} {'máx/media':>10} {'mín/media':>10} {'coef. variación':>16}")
variaciones = {}
for v in (1, 10, 100, 1000):
    anillo = AnilloConsistente(virtuales=v)
    for s in SERVIDORES[:10]:
        anillo.añadir(s)
    reparto = Counter(anillo.servidor(h) for h in hashes)
    cargas = np.array([reparto[s] for s in SERVIDORES[:10]])
    variaciones[v] = cargas.std() / cargas.mean()
    print(f"{v:9} {cargas.max() / cargas.mean():10.2f} {cargas.min() / cargas.mean():10.2f} {variaciones[v]:16.3f}")
assert variaciones[1] > variaciones[10] > variaciones[100]

capacidades = {"tenerife": 4, "gran-canaria": 4, "lanzarote": 2, "fuerteventura": 2, "el-hierro": 1}
anillo = AnilloConsistente(virtuales=100)
for s, peso in capacidades.items():
    anillo.añadir(s, peso)
reparto = Counter(anillo.servidor(h) for h in hashes)
total = sum(capacidades.values())
for s, peso in capacidades.items():
    print(f"{s:14} peso {peso / total:6.1%}  recibe {reparto[s] / len(hashes):6.1%}")
    assert abs(reparto[s] / len(hashes) - peso / total) < 0.04

virtuales  máx/media  mín/media  coef. variación
        1       2.52       0.13            0.707
       10       1.50       0.70            0.251
      100       1.16       0.80            0.108
     1000       1.06       0.96            0.028
tenerife       peso  30.8%  recibe  31.9%
gran-canaria   peso  30.8%  recibe  30.4%
lanzarote      peso  15.4%  recibe  13.6%
fuerteventura  peso  15.4%  recibe  16.4%
el-hierro      peso   7.7%  recibe   7.6%


## 14. Hashing por rendezvous

Una alternativa sin anillo es el *rendezvous hashing* (HRW, *highest random weight*): cada clave va al servidor con la mayor puntuación `hash(servidor, clave)`. No necesita nodos virtuales para equilibrar, y al añadir o quitar un servidor solo se mueven las claves que ganan o pierden frente a él; a cambio, cada consulta cuesta `O(N)` hash. Comprobar las dos propiedades de movimiento mínimo, comparar el equilibrio con el anillo de 100 nodos virtuales y contar los hash por consulta.

In [14]:
puntuaciones = {s: [hash64(f"{s}|{c}", 3) for c in claves] for s in SERVIDORES}


def rendezvous(servidores):
    return [max(servidores, key=lambda s: puntuaciones[s][i]) for i in range(len(claves))]


con_10 = rendezvous(SERVIDORES[:10])
con_11 = rendezvous(SERVIDORES[:11])
sin_lanzarote = rendezvous([s for s in SERVIDORES[:10] if s != "lanzarote"])
movidas_alta = [(a, d) for a, d in zip(con_10, con_11) if a != d]
movidas_baja = [(a, d) for a, d in zip(con_10, sin_lanzarote) if a != d]
assert all(d == SERVIDORES[10] for _, d in movidas_alta)
assert all(a == "lanzarote" for a, _ in movidas_baja) and len(movidas_baja) == con_10.count("lanzarote")
print(f"Al añadir un servidor se mueven {len(movidas_alta) / len(claves):.1%} (esperado {1 / 11:.1%}), todas hacia él")
print(f"Al quitar lanzarote se mueven {len(movidas_baja)} claves, solo las suyas")

reparto_hrw = Counter(con_10)
cargas_hrw = np.array([reparto_hrw[s] for s in SERVIDORES[:10]])
anillo = AnilloConsistente(virtuales=100)
for s in SERVIDORES[:10]:
    anillo.añadir(s)
reparto_anillo = Counter(anillo.servidor(h) for h in hashes)
cargas_anillo = np.array([reparto_anillo[s] for s in SERVIDORES[:10]])
ruido = math.sqrt(len(claves) / 10 * (1 - 1 / 10)) / (len(claves) / 10)
print(f"Coeficiente de variación: rendezvous {cargas_hrw.std() / cargas_hrw.mean():.3f}, "
      f"anillo {cargas_anillo.std() / cargas_anillo.mean():.3f}, ruido binomial {ruido:.3f}")
print(f"Hash por consulta: rendezvous {10}, anillo 1 (más una búsqueda binaria entre {len(anillo.puntos)} puntos)")
assert cargas_hrw.std() / cargas_hrw.mean() < 2 * ruido

Al añadir un servidor se mueven 9.0% (esperado 9.1%), todas hacia él


Al quitar lanzarote se mueven 1955 claves, solo las suyas
Coeficiente de variación: rendezvous 0.016, anillo 0.108, ruido binomial 0.021
Hash por consulta: rendezvous 10, anillo 1 (más una búsqueda binaria entre 1000 puntos)
